# Declarations and Initializations

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, ScalarFormatter

from scipy.signal import savgol_filter#, correlate, correlation_lags

#from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    #accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.compose import TransformedTargetRegressor
#from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    max_error,
)
#from sklearn.pipeline import Pipeline
#from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    #cross_val_predict,
    #cross_validate,
    #StratifiedGroupKFold,
    StratifiedKFold,
    #train_test_split,
)
from sklearn.pipeline import Pipeline

from tsfresh import extract_features

In [ ]:
FILE_TITLE = 'Combined-2'
DATA_PATH = f"../data/02_preprocessed/{FILE_TITLE}/"

In [ ]:
DATA = pd.DataFrame()
file_list = os.listdir(DATA_PATH)

for f in file_list:
    if ".csv" in f:
        trial_file = pd.read_csv(f"{DATA_PATH}{f}").dropna()
        DATA = pd.concat([DATA , trial_file] , ignore_index = True)

MATERIALS = list(DATA['Sample'].unique())

MATERIAL_GROUPS = {
'bismuth': 'metal',
'nickel': 'metal',
'titanium': 'metal',
'iron': 'metal',
'aluminum': 'metal',
'copper': 'metal',

'cement': 'ceramic',
'gypsum': 'ceramic',
'carbon': 'ceramic', 'graphite': 'ceramic',

'cork_coarse': 'composite',
'cork_fine': 'composite',
'wood': 'composite',
'abrasive': 'composite',

'pdms': 'polymer',

'ps_foam': 'foam',
'pu_foam': 'foam'}

DATA['Label']=DATA['Sample'].apply(lambda x : MATERIAL_GROUPS[x])
DATA["eff"] = (DATA["k"] * DATA["rho"] * DATA["cp"]) ** 0.5

# Determination of elbow `time`, `value` , and `index`

**Function name:**
- find_first_elbow()

**Returned values:**
- time[elbow_position]
- signal[elbow_position]
- indices[elbow_position]
- time
- smooth_signal

In [ ]:
def find_first_elbow(
    subset,
    time_col="Time",
    signal_col="Primary",
    index_col="index",
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    direction="drop",
    skip_index=5,
):
    df = subset.sort_values(time_col, kind="stable").reset_index(drop=True)

    indices = (
        df[index_col].to_numpy()
        if index_col in df.columns
        else np.arange(len(df))
    )

    df = df.iloc[skip_index:]
    indices = indices[skip_index:]

    time = df[time_col].to_numpy(dtype=float)
    signal = df[signal_col].to_numpy(dtype=float)

    valid = np.isfinite(time) & np.isfinite(signal)
    time = time[valid]
    signal = signal[valid]
    indices = indices[valid]

    n = len(signal)

    if n < 3:
        return np.nan, np.nan, None, time, signal

    window = min(smooth_window, n)

    if window % 2 == 0:
        window -= 1

    if window <= polyorder:
        window = polyorder + 1

        if window % 2 == 0:
            window += 1

    if window > n:
        return np.nan, np.nan, None, time, signal

    smooth_signal = savgol_filter(
        signal,
        window_length=window,
        polyorder=polyorder,
    )

    derivative = np.gradient(smooth_signal, time)

    if direction == "drop":
        strongest_idx = np.argmin(derivative)
        active = derivative < threshold_frac * derivative[strongest_idx]

    elif direction == "rise":
        strongest_idx = np.argmax(derivative)
        active = derivative > threshold_frac * derivative[strongest_idx]

    else:
        raise ValueError("direction must be 'drop' or 'rise'")

    elbow_position = 0

    for position in range(strongest_idx, -1, -1):
        if not active[position]:
            elbow_position = position + 1
            break

    return (
        time[elbow_position],
        signal[elbow_position],
        indices[elbow_position],
        time,
        smooth_signal,
    )

## Storage of elbow `time`, `value` , and `index` into df.DATA

In [ ]:
ELBOW_IDX_DICT = {}
DATA["Elbow time"] = np.nan
DATA["Elbow value"] = np.nan
DATA["Elbow index"] = np.nan

for material in MATERIALS:
    current_material = DATA[DATA['Sample'] == material]
    current_material = current_material.sort_values(["Trial", "Time"])

    trial_numbers = sorted(current_material["Trial"].unique())

    ELBOW_IDX_DICT[material] = {}

    for trial in trial_numbers:
        trial = int(trial)
        subset = current_material[current_material["Trial"] == trial]

        (
            ELBOW_TIME,
            ELBOW_VALUE,
            ELBOW_IDX,
            SMOOTH_TIME,
            SMOOTH_PRIMARY,
        ) = find_first_elbow(subset)

        ELBOW_IDX_DICT[material][trial] = int(ELBOW_IDX)

        row_condition = ((DATA["Sample"] == material) & (DATA["Trial"] == trial))

        DATA.loc[row_condition, "Elbow time"]   = ELBOW_TIME
        DATA.loc[row_condition, "Elbow value"]  = ELBOW_VALUE
        DATA.loc[row_condition, "Elbow index"]  = ELBOW_IDX

## Visual inspection of elbow

In [ ]:
for material in MATERIALS:
    current_material = DATA[DATA["Sample"] == material]
    trial_numbers = sorted(current_material["Trial"].unique())

    n_cols = min(3, len(trial_numbers))
    n_rows = math.ceil(len(trial_numbers) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trial_numbers):
        subset = (
            current_material[current_material["Trial"].astype(int) == trial]
            .sort_values("Time")
            .copy()
        )

        elbow_time = subset["Elbow time"].iloc[0]
        elbow_value = subset["Elbow value"].iloc[0]
        elbow_idx = subset["Elbow index"].iloc[0]

        ax.plot(
            subset["Time"],
            subset["Primary"],
            label="Primary",
        )

        ax.plot(
            subset["Time"],
            subset["Secondary"],
            label="Secondary",
        )

        if np.isfinite(elbow_time):
            ax.axvline(
                elbow_time,
                linestyle=":",
                label="Elbow",
            )

            ax.scatter(
                elbow_time,
                elbow_value,
                zorder=5,
            )

        ax.set(
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
            xlabel="Time",
            ylabel="Value",
        )

        ax.legend(fontsize=8)

        print(
            f"{material} - Trial {trial} - "
            f"elbow index: {elbow_idx}, elbow time: {elbow_time:.3f}"
        )

    for ax in axes[len(trial_numbers):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} elbow visual inspection",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Elbow Position Normalization

In [ ]:
DATA

In [ ]:
DATA["Time from elbow"] = DATA["Time"] - DATA["Elbow time"]
DATA["Primary aligned"] = DATA["Primary"] - DATA["Elbow value"]
DATA["Secondary aligned"] = DATA["Secondary"] - DATA["Elbow value"]
DATA = DATA[(DATA['Time from elbow'] >= 0) & (DATA['Time from elbow'] < 5.1)]


# Feature extraction

## Thermal Features

In [ ]:
def extract_trial_features(
    trial_data,
    analysis_window=(0.0, 5.0),
    early_window=(0.0, 1.0),
    mid_window=(1.0, 3.0),
    late_window=(3.0, 5.0),
    smooth=True,
    savgol_window=11,
    savgol_polyorder=2,
    k_pdms=0.15,             # W/(m·K), approximate PDMS thermal conductivity
    pdms_thickness_m=1e-3,   # m, replace with your actual PDMS layer thickness
    contact_area_m2=1e-6,    # m², 1 mm × 1 mm
    effective_depth_m=1e-3,  # m, assumed thermal penetration depth for rho/cp features
):
    """
    Extract cleaned thermal-response features from one trial.

    Required columns
    ----------------
    Time
    Primary
    Secondary

    Optional columns
    ----------------
    rho : literature density, kg/m³
    cp  : literature specific heat capacity, J/(kg·K)

    Assumptions
    -----------
    - First row is already the detected elbow/contact onset.
    - Primary and Secondary are temperature values, or are linearly proportional to temperature.
    - Time is in seconds.
    - Primary is the top sensor.
    - Secondary is the bottom sensor.
    - difference = Primary - Secondary.
    """

    # ------------------------------------------------------------
    # Helper functions
    # ------------------------------------------------------------

    trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

    def make_mask(time_array, window):
        start, end = window
        return (time_array >= start) & (time_array <= end)

    def finite_first(series):
        values = pd.to_numeric(series, errors="coerce").dropna()
        if len(values) == 0:
            return np.nan
        return float(values.iloc[0])

    def safe_ratio(numerator, denominator):
        if (
            not np.isfinite(numerator)
            or not np.isfinite(denominator)
            or np.isclose(denominator, 0.0)
        ):
            return np.nan
        return float(numerator / denominator)

    def fitted_slope(time_array, signal_array, window):
        """
        Linear-fit slope within a selected time window.
        """
        mask = make_mask(time_array, window)

        valid = (
            mask
            & np.isfinite(time_array)
            & np.isfinite(signal_array)
        )

        if valid.sum() < 3:
            return np.nan

        slope, _ = np.polyfit(
            time_array[valid],
            signal_array[valid],
            deg=1,
        )

        return float(slope)

    def smooth_signal(signal):
        signal = np.asarray(signal, dtype=float)

        if not smooth:
            return signal.copy()

        window = min(int(savgol_window), len(signal))

        # Savitzky-Golay window must be odd.
        if window % 2 == 0:
            window -= 1

        minimum_window = savgol_polyorder + 2

        if minimum_window % 2 == 0:
            minimum_window += 1

        if window < minimum_window:
            return signal.copy()

        return savgol_filter(
            signal,
            window_length=window,
            polyorder=savgol_polyorder,
            mode="interp",
        )

    def extract_signal_features(name, signal, rate, time_array, result):
        """
        Extract selected 0th-order and 1st-order signal features.
        """
        baseline = float(signal[0])
        change = signal - baseline

        # 0th-order signal features
        result[f"{name}_max_change"] = float(change[-1])

        result[f"{name}_response_auc"] = float(
            trapz(
                np.abs(change),
                time_array,
            )
        )

        # 1st-order derivative features
        result[f"{name}_early_window_rate"] = fitted_slope(
            time_array,
            signal,
            early_window,
        )

        result[f"{name}_mid_window_rate"] = fitted_slope(
            time_array,
            signal,
            mid_window,
        )

        result[f"{name}_late_window_rate"] = fitted_slope(
            time_array,
            signal,
            late_window,
        )

        result[f"{name}_max_abs_rate"] = float(
            np.nanmax(np.abs(rate))
        )

        result[f"{name}_rate_auc"] = float(
            trapz(
                np.abs(rate),
                time_array,
            )
        )

        result[f"{name}_rate_std"] = (
            float(np.nanstd(rate, ddof=1))
            if len(rate) > 1
            else np.nan
        )

        return result

    # ------------------------------------------------------------
    # 1. Clean and sort trial data
    # ------------------------------------------------------------

    required_columns = ["Time", "Primary", "Secondary"]
    optional_columns = ["rho", "cp"]

    missing_columns = set(required_columns) - set(trial_data.columns)

    if missing_columns:
        raise KeyError(f"Missing columns: {sorted(missing_columns)}")

    available_columns = required_columns + [
        col for col in optional_columns
        if col in trial_data.columns
    ]

    trial_data = trial_data[available_columns].copy()

    for col in available_columns:
        trial_data[col] = pd.to_numeric(
            trial_data[col],
            errors="coerce",
        )

    # Literature property values for this trial
    rho = (
        finite_first(trial_data["rho"])
        if "rho" in trial_data.columns
        else np.nan
    )

    cp = (
        finite_first(trial_data["cp"])
        if "cp" in trial_data.columns
        else np.nan
    )

    rho_cp = (
        rho * cp
        if np.isfinite(rho) and np.isfinite(cp)
        else np.nan
    )

    trial_data = (
        trial_data
        .dropna(subset=required_columns)
        .sort_values("Time")
        .drop_duplicates(subset="Time")
        .reset_index(drop=True)
    )

    if len(trial_data) < 4:
        raise ValueError("The trial contains too few valid observations.")

    original_time = trial_data["Time"].to_numpy(dtype=float)
    primary = trial_data["Primary"].to_numpy(dtype=float)
    secondary = trial_data["Secondary"].to_numpy(dtype=float)

    if np.any(np.diff(original_time) <= 0):
        raise ValueError("Time must be strictly increasing.")

    # ------------------------------------------------------------
    # 2. Treat first row as elbow/contact onset
    # ------------------------------------------------------------

    elbow_time = float(original_time[0])
    time = original_time - elbow_time

    # ------------------------------------------------------------
    # 3. Restrict to fixed analysis window
    # ------------------------------------------------------------

    analysis_mask = make_mask(
        time,
        analysis_window,
    )

    time = time[analysis_mask]
    primary = primary[analysis_mask]
    secondary = secondary[analysis_mask]

    if len(time) < 4:
        raise ValueError("Too few observations inside analysis_window.")

    # ------------------------------------------------------------
    # 4. Smooth signals
    # ------------------------------------------------------------

    primary_smooth = smooth_signal(primary)
    secondary_smooth = smooth_signal(secondary)

    # Top-minus-bottom convention
    difference_smooth = primary_smooth - secondary_smooth

    # ------------------------------------------------------------
    # 5. First derivatives
    # ------------------------------------------------------------

    primary_rate = np.gradient(
        primary_smooth,
        time,
    )

    secondary_rate = np.gradient(
        secondary_smooth,
        time,
    )

    difference_rate = primary_rate - secondary_rate

    # ------------------------------------------------------------
    # 6. Initialize result
    # ------------------------------------------------------------

    result = {
        "elbow_time": elbow_time,
        "analysis_start": analysis_window[0],
        "analysis_end": analysis_window[1],

        # literature / geometry metadata
        "rho_lit": rho,
        "cp_lit": cp,
        "rho_cp_lit": rho_cp,
        "contact_area_m2": contact_area_m2,
        "effective_depth_m": effective_depth_m,
        "thermal_capacitance_per_area_lit": (
            rho_cp * effective_depth_m
            if np.isfinite(rho_cp)
            else np.nan
        ),
    }

    # ------------------------------------------------------------
    # 7. Primary, Secondary, and Difference signal features
    # ------------------------------------------------------------

    signals = {
        "primary": {
            "signal": primary_smooth,
            "rate": primary_rate,
        },
        "secondary": {
            "signal": secondary_smooth,
            "rate": secondary_rate,
        },
        "difference": {
            "signal": difference_smooth,
            "rate": difference_rate,
        },
    }

    for name, values in signals.items():
        result = extract_signal_features(
            name=name,
            signal=values["signal"],
            rate=values["rate"],
            time_array=time,
            result=result,
        )

    # ------------------------------------------------------------
    # 8. Secondary-to-primary rate ratios
    # ------------------------------------------------------------

    result["secondary_early_rate_ratio"] = safe_ratio(
        result["secondary_early_window_rate"],
        result["primary_early_window_rate"],
    )

    result["secondary_mid_rate_ratio"] = safe_ratio(
        result["secondary_mid_window_rate"],
        result["primary_mid_window_rate"],
    )

    result["secondary_late_rate_ratio"] = safe_ratio(
        result["secondary_late_window_rate"],
        result["primary_late_window_rate"],
    )

    result["secondary_rate_auc_ratio"] = safe_ratio(
        result["secondary_rate_auc"],
        result["primary_rate_auc"],
    )

    # ------------------------------------------------------------
    # 9. Top-bottom rate-difference features
    # ------------------------------------------------------------

    result["mean_top_bottom_rate_difference"] = float(
        np.nanmean(difference_rate)
    )

    result["max_top_bottom_rate_difference"] = float(
        np.nanmax(difference_rate)
    )

    result["max_abs_top_bottom_rate_difference"] = float(
        np.nanmax(np.abs(difference_rate))
    )

    result["top_bottom_rate_difference_auc"] = float(
        trapz(
            np.abs(difference_rate),
            time,
        )
    )

    result["top_bottom_rate_difference_std"] = (
        float(np.nanstd(difference_rate, ddof=1))
        if len(difference_rate) > 1
        else np.nan
    )

    # ------------------------------------------------------------
    # 10. PDMS heat-flux features
    # q''_PDMS = k_PDMS / L_PDMS * (T_top - T_bottom)
    # ------------------------------------------------------------

    q_pdms = (
        (k_pdms / pdms_thickness_m)
        * difference_smooth
    )

    q_pdms_rate = np.gradient(
        q_pdms,
        time,
    )

    result["q_pdms_mean"] = float(
        np.nanmean(q_pdms)
    )

    result["q_pdms_max"] = float(
        np.nanmax(q_pdms)
    )

    result["q_pdms_min"] = float(
        np.nanmin(q_pdms)
    )

    result["q_pdms_max_abs"] = float(
        np.nanmax(np.abs(q_pdms))
    )

    result["q_pdms_final"] = float(
        q_pdms[-1] - q_pdms[0]
    )

    result["q_pdms_auc"] = float(
        trapz(
            np.abs(q_pdms),
            time,
        )
    )

    result["q_pdms_early_window_rate"] = fitted_slope(
        time,
        q_pdms,
        early_window,
    )

    result["q_pdms_mid_window_rate"] = fitted_slope(
        time,
        q_pdms,
        mid_window,
    )

    result["q_pdms_late_window_rate"] = fitted_slope(
        time,
        q_pdms,
        late_window,
    )

    result["q_pdms_rate_auc"] = float(
        trapz(
            np.abs(q_pdms_rate),
            time,
        )
    )

    result["q_pdms_rate_std"] = (
        float(np.nanstd(q_pdms_rate, ddof=1))
        if len(q_pdms_rate) > 1
        else np.nan
    )

    # ------------------------------------------------------------
    # 11. Literature rho/cp-based features
    # These are reference-informed, not independently measured.
    # ------------------------------------------------------------

    if np.isfinite(rho_cp):

        # Energy density from primary temperature response:
        # u = rho * cp * ΔT
        result["primary_energy_density_max_change"] = (
            rho_cp * result["primary_max_change"]
        )

        result["primary_energy_density_response_auc"] = (
            rho_cp * result["primary_response_auc"]
        )

        # Power density from primary temperature rate:
        # p_v = rho * cp * dT/dt
        result["primary_power_density_early_window_rate"] = (
            rho_cp * result["primary_early_window_rate"]
        )

        result["primary_power_density_mid_window_rate"] = (
            rho_cp * result["primary_mid_window_rate"]
        )

        result["primary_power_density_late_window_rate"] = (
            rho_cp * result["primary_late_window_rate"]
        )

        result["primary_power_density_max_abs_rate"] = (
            rho_cp * result["primary_max_abs_rate"]
        )

        result["primary_power_density_rate_auc"] = (
            rho_cp * result["primary_rate_auc"]
        )

        # Apparent heat flux using assumed effective heated depth:
        # q''_app = rho * cp * L_eff * dT/dt
        result["primary_q_app_early_window_rate"] = (
            rho_cp
            * effective_depth_m
            * result["primary_early_window_rate"]
        )

        result["primary_q_app_mid_window_rate"] = (
            rho_cp
            * effective_depth_m
            * result["primary_mid_window_rate"]
        )

        result["primary_q_app_late_window_rate"] = (
            rho_cp
            * effective_depth_m
            * result["primary_late_window_rate"]
        )

        result["primary_q_app_max_abs_rate"] = (
            rho_cp
            * effective_depth_m
            * result["primary_max_abs_rate"]
        )

        result["primary_q_app_rate_auc"] = (
            rho_cp
            * effective_depth_m
            * result["primary_rate_auc"]
        )

    else:
        rho_cp_feature_names = [
            "primary_energy_density_max_change",
            "primary_energy_density_response_auc",
            "primary_power_density_early_window_rate",
            "primary_power_density_mid_window_rate",
            "primary_power_density_late_window_rate",
            "primary_power_density_max_abs_rate",
            "primary_power_density_rate_auc",
            "primary_q_app_early_window_rate",
            "primary_q_app_mid_window_rate",
            "primary_q_app_late_window_rate",
            "primary_q_app_max_abs_rate",
            "primary_q_app_rate_auc",
        ]

        for feature_name in rho_cp_feature_names:
            result[feature_name] = np.nan

    return pd.Series(result)

In [ ]:
# ------------------------------------------------------------
# Run feature extraction
# ------------------------------------------------------------

group_columns = [
    "Sample",
    "Trial",
    "Label",
    "k",
    "eff"
]

FEATURES_THERMAL = (
    DATA
    .groupby(
        group_columns,
        dropna=False,
    )
    .apply(
        lambda trial_data: extract_trial_features(
            trial_data,
            analysis_window=(0.0, 5.0),
            early_window=(0.0, 1.0),
            mid_window=(1.0, 3.0),
            late_window=(3.0, 5.0),

            # Replace these with your actual sensor-stack values
            k_pdms=0.15,
            pdms_thickness_m=7e-4,

            # contact area: 1 mm × 1 mm
            contact_area_m2=1e-6,

            # Assumed effective thermal penetration depth
            effective_depth_m=1e-3,
        )
    )
    .reset_index()
)

FEATURES_THERMAL.head()

In [ ]:
# ============================================================
# 1. Select the extracted features to visualize
# ============================================================

features = [
    # --------------------------------------------------------
    # Primary sensor: 0th-order and 1st-order features
    # --------------------------------------------------------
    "primary_max_change",
    "primary_response_auc",
    "primary_early_window_rate",
    "primary_mid_window_rate",
    "primary_late_window_rate",
    "primary_max_abs_rate",
    "primary_rate_auc",
    "primary_rate_std",

    # --------------------------------------------------------
    # Secondary sensor: 0th-order and 1st-order features
    # --------------------------------------------------------
    "secondary_max_change",
    "secondary_response_auc",
    "secondary_early_window_rate",
    "secondary_mid_window_rate",
    "secondary_late_window_rate",
    "secondary_max_abs_rate",
    "secondary_rate_auc",
    "secondary_rate_std",

    # --------------------------------------------------------
    # Primary - Secondary difference
    # --------------------------------------------------------
    "difference_max_change",
    "difference_response_auc",
    "difference_early_window_rate",
    "difference_mid_window_rate",
    "difference_late_window_rate",
    "difference_max_abs_rate",
    "difference_rate_auc",
    "difference_rate_std",

    # --------------------------------------------------------
    # Secondary-to-primary ratios
    # --------------------------------------------------------
    "secondary_early_rate_ratio",
    "secondary_mid_rate_ratio",
    "secondary_late_rate_ratio",
    "secondary_rate_auc_ratio",

    # --------------------------------------------------------
    # Cross-sensor derivative features
    # --------------------------------------------------------
    "mean_top_bottom_rate_difference",
    "max_top_bottom_rate_difference",
    "max_abs_top_bottom_rate_difference",
    "top_bottom_rate_difference_auc",
    "top_bottom_rate_difference_std",

    # --------------------------------------------------------
    # PDMS heat-flux features
    # q''_PDMS = k_PDMS / L_PDMS * (Primary - Secondary)
    # --------------------------------------------------------
    "q_pdms_mean",
    "q_pdms_max",
    "q_pdms_min",
    "q_pdms_max_abs",
    "q_pdms_final",
    "q_pdms_auc",
    "q_pdms_early_window_rate",
    "q_pdms_mid_window_rate",
    "q_pdms_late_window_rate",
    "q_pdms_rate_auc",
    "q_pdms_rate_std",

    # --------------------------------------------------------
    # Literature rho/cp-based primary energy-density features
    # --------------------------------------------------------
    "primary_energy_density_max_change",
    "primary_energy_density_response_auc",

    # --------------------------------------------------------
    # Literature rho/cp-based primary power-density features
    # --------------------------------------------------------
    "primary_power_density_early_window_rate",
    "primary_power_density_mid_window_rate",
    "primary_power_density_late_window_rate",
    "primary_power_density_max_abs_rate",
    "primary_power_density_rate_auc",

    # --------------------------------------------------------
    # Literature rho/cp-based apparent heat-flux features
    # --------------------------------------------------------
    "primary_q_app_early_window_rate",
    "primary_q_app_mid_window_rate",
    "primary_q_app_late_window_rate",
    "primary_q_app_max_abs_rate",
    "primary_q_app_rate_auc",
]


# Keep only features that actually exist in features_df.
# This prevents KeyError if some names were not generated.
features = [
    feature
    for feature in features
    if feature in FEATURES_THERMAL.columns
]

print(f"Number of features to plot: {len(features)}")

In [ ]:
legend_col = "Sample"

plots_per_fig = 3
n_cols = 3
n_rows = 1

n_figures = math.ceil(
    len(features) / plots_per_fig
)

desired_order = [
    "copper",
    "aluminum",
    "nickel",
    "iron",
    "titanium",
    "bismuth",
    "carbon",
    "cement",
    "gypsum",
    "pdms",
    "wood",
    "cork_fine",
    "cork_coarse",
    "pu_foam",
    "ps_foam",
    "abrasive",      # Polyester foam
]

available_groups = (
    FEATURES_THERMAL[legend_col]
    .dropna()
    .unique()
)

groups = [
    group
    for group in desired_order
    if group in available_groups
]

# Include any samples not listed in desired_order
remaining_groups = [
    group
    for group in available_groups
    if group not in groups
]

groups.extend(remaining_groups)


for fig_num in range(n_figures):

    start = fig_num * plots_per_fig
    end = start + plots_per_fig

    current_features = features[start:end]

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 5),
    )

    axes = np.atleast_1d(axes).flatten()

    for ax, feature in zip(
        axes,
        current_features,
    ):

        for group_idx, group in enumerate(groups):

            subset = FEATURES_THERMAL[
                FEATURES_THERMAL[legend_col] == group
            ]

            values = pd.to_numeric(
                subset[feature],
                errors="coerce",
            ).dropna()

            if values.empty:
                continue

            y_position = np.full(
                len(values),
                group_idx,
                dtype=float,
            )

            # Small vertical jitter helps reveal overlapping trials
            jitter = np.random.uniform(
                0,
                0.005,
                size=len(values),
            )

            ax.scatter(
                values,
                y_position + jitter,
                label=str(group),
                alpha=0.75,
                s=60,
            )

        ax.set_yticks(
            range(len(groups))
        )

        ax.set_yticklabels(groups)

        ax.set_xlabel(feature)
        ax.set_ylabel(legend_col)
        ax.set_title(feature)

        ax.grid(
            axis="x",
            linestyle=":",
            alpha=0.5,
        )

    # Hide unused axes in the final figure
    for ax in axes[len(current_features):]:
        ax.axis("off")

    """
    fig.suptitle(
        (
            "Feature number-line plots — "
            f"Figure {fig_num + 1}"
        ),
        fontsize=16,
    )
    """

    plt.tight_layout(
        rect=[0, 0, 1, 0.95]
    )

    plt.show()

In [ ]:
legend_col = "Label"
y_col = "k"
plots_per_fig = 3

# --------------------------------------------------
# Prepare plotting data
# --------------------------------------------------

plot_df = (
    FEATURES_THERMAL
    .copy()
    .replace([np.inf, -np.inf], np.nan)
)

plot_df[y_col] = pd.to_numeric(
    plot_df[y_col],
    errors="coerce"
)

# Logarithmic axes require positive values
plot_df = plot_df[
    plot_df[y_col] > 0
].copy()


# --------------------------------------------------
# Keep features containing valid numeric values
# --------------------------------------------------

valid_features = []

for feature in features:

    if feature not in plot_df.columns:
        print(f"Missing feature: {feature}")
        continue

    plot_df[feature] = pd.to_numeric(
        plot_df[feature],
        errors="coerce"
    )

    if plot_df[[feature, y_col]].dropna().empty:
        print(f"No valid numeric values: {feature}")
        continue

    valid_features.append(feature)


# --------------------------------------------------
# Order labels automatically by median k
# --------------------------------------------------

groups = (
    plot_df
    .dropna(subset=[legend_col])
    .groupby(legend_col)[y_col]
    .median()
    .sort_values()
    .index
    .tolist()
)


# --------------------------------------------------
# Assign one consistent color to each label
# --------------------------------------------------

cmap = plt.get_cmap(
    "tab20",
    len(groups)
)

group_colors = {
    group: cmap(i)
    for i, group in enumerate(groups)
}


# --------------------------------------------------
# Create plots
# --------------------------------------------------

n_figures = math.ceil(
    len(valid_features) / plots_per_fig
)

for fig_num in range(n_figures):

    current_features = valid_features[
        fig_num * plots_per_fig:
        (fig_num + 1) * plots_per_fig
    ]

    n_current = len(current_features)

    fig, axes = plt.subplots(
        1,
        n_current,
        figsize=(6 * n_current, 8),
        squeeze=False
    )

    axes = axes.ravel()

    for ax, feature in zip(
        axes,
        current_features
    ):

        for group in groups:

            subset = plot_df[
                plot_df[legend_col] == group
            ].dropna(
                subset=[feature, y_col]
            )

            if subset.empty:
                continue

            ax.scatter(
                subset[feature],
                subset[y_col],
                color=group_colors[group],
                alpha=0.8,
                s=60
            )

        ax.set_xlabel(feature)

        ax.set_ylabel(
            "Thermal conductivity, k [W/(m·K)]"
        )

        ax.set_title(
            f"{feature} vs k"
        )

        # Logarithmic numeric y-axis
        ax.set_yscale("log")

        # Major ticks at powers of 10
        ax.yaxis.set_major_locator(
            LogLocator(base=10)
        )

        # Display 0.01, 0.1, 1, 10, 100 instead of scientific notation
        ax.yaxis.set_major_formatter(
            FuncFormatter(
                lambda value, position: f"{value:g}"
            )
        )

        ax.grid(
            which="both",
            linestyle=":",
            alpha=0.5
        )


    # --------------------------------------------------
    # Shared legend
    # --------------------------------------------------

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor=group_colors[group],
            markeredgecolor=group_colors[group],
            markersize=7,
            label=str(group).replace("_", " ").title()
        )
        for group in groups
    ]

    fig.legend(
        handles=legend_handles,
        title="Material cluster",
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=min(8, len(groups)),
        fontsize=8
    )

    plt.tight_layout(
        rect=[0, 0.14, 1, 1]
    )

    plt.show()

In [ ]:
legend_col = "Label"

plots_per_fig = 3
n_cols = 3
n_rows = 1

n_figures = math.ceil(
    len(features) / plots_per_fig
)

desired_order = [
    'metal'
    ,'ceramic'
    ,'polymer'
    ,'composite'
]

available_groups = (
    FEATURES_THERMAL[legend_col]
    .dropna()
    .unique()
)

groups = [
    group
    for group in desired_order
    if group in available_groups
]

# Include any samples not listed in desired_order
remaining_groups = [
    group
    for group in available_groups
    if group not in groups
]

groups.extend(remaining_groups)


for fig_num in range(n_figures):

    start = fig_num * plots_per_fig
    end = start + plots_per_fig

    current_features = features[start:end]

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 5),
    )

    axes = np.atleast_1d(axes).flatten()

    for ax, feature in zip(
        axes,
        current_features,
    ):

        for group_idx, group in enumerate(groups):

            subset = FEATURES_THERMAL[
                FEATURES_THERMAL[legend_col] == group
            ]

            values = pd.to_numeric(
                subset[feature],
                errors="coerce",
            ).dropna()

            if values.empty:
                continue

            y_position = np.full(
                len(values),
                group_idx,
                dtype=float,
            )

            # Small vertical jitter helps reveal overlapping trials
            jitter = np.random.uniform(
                0,
                0.005,
                size=len(values),
            )

            ax.scatter(
                values,
                y_position + jitter,
                label=str(group),
                alpha=0.75,
                s=60,
            )

        ax.set_yticks(
            range(len(groups))
        )

        ax.set_yticklabels(groups)

        ax.set_xlabel(feature)
        ax.set_ylabel(legend_col)
        ax.set_title(feature)

        ax.grid(
            axis="x",
            linestyle=":",
            alpha=0.5,
        )

    # Hide unused axes in the final figure
    for ax in axes[len(current_features):]:
        ax.axis("off")

    fig.suptitle(
        (
            "Feature number-line plots — "
            f"Figure {fig_num + 1}"
        ),
        fontsize=16,
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.95]
    )

    plt.show()

## `tsfresh` Features

In [ ]:
_basic = [
    "mean", "median", "minimum", "maximum", "absolute_maximum"
]

_variability = [
    "standard_deviation", "variance", "root_mean_square",
    "mean_abs_change", "mean_change"
]

_distribution = ["skewness", "kurtosis"]
_energy = ["abs_energy", "sum_values"]
_counts = ["count_above_mean", "count_below_mean"]
_strikes = ["longest_strike_above_mean", "longest_strike_below_mean"]

_linear_trend = [
    {"attr": attr}
    for attr in ("slope", "intercept", "rvalue", "pvalue", "stderr")
]

_autocorr = [
    {"lag": lag}
    for lag in (1, 2, 5, 10)
]

_change_quantiles = [
    {"ql": 0.0, "qh": 0.2, "isabs": False, "f_agg": "mean"},
    {"ql": 0.0, "qh": 0.2, "isabs": True,  "f_agg": "mean"},
    {"ql": 0.8, "qh": 1.0, "isabs": False, "f_agg": "mean"},
    {"ql": 0.8, "qh": 1.0, "isabs": True,  "f_agg": "mean"},
]

_cid_ce = [
    {"normalize": True},
    {"normalize": False},
]

settings = {
    feature_name: None
    for feature_name in (
        _basic
        + _variability
        + _distribution
        + _energy
        + _counts
        + _strikes
    )
}

settings.update({
    "linear_trend": _linear_trend,
    "autocorrelation": _autocorr,
    "change_quantiles": _change_quantiles,
    "cid_ce": _cid_ce,
})

In [ ]:
DATA['diff'] = DATA['Secondary'] - DATA['Primary']

In [ ]:
# use only filtered data
ts_data = DATA.copy()

# create one ID per material-trial
ts_data["id"] = (
    ts_data["Sample"].astype(str) +
    "_trial_" +
    ts_data["Trial"].astype(int).astype(str)
)

# primary sensor table
primary = ts_data[[
    "id",
    "Time",
    "Primary",
    "k",
    "eff"
]].copy()

primary = primary.rename(columns={
    "Time": "time",
    "Primary": "value"
})

primary["kind"] = "Primary"

# secondary sensor table
secondary = ts_data[[
    "id",
    "Time",
    "Secondary"
]].copy()

secondary = secondary.rename(columns={
    "Time": "time",
    "Secondary": "value"
})

secondary["kind"] = "Secondary"


# diff sensor table
diff = ts_data[[
    "id",
    "Time",
    "diff"
]].copy()

diff = diff.rename(columns={
    "Time": "time",
    "diff": "value"
})

diff["kind"] = "Difference"

# combine primary and secondary
tsfresh_input = pd.concat(
    [primary, secondary, diff],
    ignore_index=True
)

tsfresh_input = tsfresh_input[[
    "id",
    "time",
    "kind",
    "value",
    "k"
]]

# clean invalid values
tsfresh_input = tsfresh_input.replace([np.inf, -np.inf], np.nan)
tsfresh_input = tsfresh_input.dropna()

In [ ]:
FEATURES_TSFRESH = extract_features(
    tsfresh_input,
    column_id="id",
    column_sort="time",
    column_kind="kind",
    column_value="value",
    default_fc_parameters=settings,
    n_jobs=0
)

In [ ]:
labels = (
    ts_data[["id", "Sample", "Trial", "Label", "eff"]]
    .drop_duplicates()
    .set_index("id")
)
FEATURES_TSFRESH = FEATURES_TSFRESH.dropna(axis=1)


FEATURES_TSFRESH = FEATURES_TSFRESH.join(labels)
FEATURES_TSFRESH = FEATURES_TSFRESH.reset_index()
#FEATURES_TSFRESH = FEATURES_TSFRESH.join(ts)
FEATURES_TSFRESH.head()

In [ ]:
# -index ; - trial

legend_col = "Sample"

# Automatically select numeric features,
# excluding identifiers and elbow_time
excluded_cols = ["Sample", "Trial", "index"]

features = [
    col
    for col in FEATURES_TSFRESH.select_dtypes(include=np.number).columns
    if col not in excluded_cols
]

plots_per_fig = 9
n_cols = 3
n_rows = 3

n_figures = math.ceil(len(features) / plots_per_fig)

# Keep the same sample order in every subplot
groups = FEATURES_TSFRESH[legend_col].dropna().unique()

for fig_num in range(n_figures):

    start = fig_num * plots_per_fig
    end = start + plots_per_fig
    current_features = features[start:end]

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 15)
    )

    axes = axes.flatten()

    for ax, feature in zip(axes, current_features):

        for group_idx, group in enumerate(groups):

            subset = FEATURES_TSFRESH[
                FEATURES_TSFRESH[legend_col] == group
            ]

            # Convert to numeric and remove NaN values
            values = pd.to_numeric(
                subset[feature],
                errors="coerce"
            ).dropna()

            # Place every sample on its own horizontal level
            y_position = np.full(
                len(values),
                group_idx
            )

            ax.scatter(
                values,
                y_position,
                label=str(group),
                alpha=0.75,
                s=60
            )

        ax.set_yticks(range(len(groups)))
        ax.set_yticklabels(groups)

        ax.set_xlabel(feature)
        ax.set_ylabel(legend_col)
        ax.set_title(feature)

        ax.grid(
            axis="x",
            linestyle=":",
            alpha=0.5
        )

    # Hide unused plots in the final figure
    for ax in axes[len(current_features):]:
        ax.axis("off")

    fig.suptitle(
        f"Feature number-line plots — Figure {fig_num + 1}",
        fontsize=16
    )

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# Create lookup table for thermal conductivity
k_lookup = (
    DATA[["Sample", "k"]]
    .dropna(subset=["Sample", "k"])
    .drop_duplicates(subset=["Sample"])
)

# Remove old k column if it exists
FEATURES_TSFRESH = FEATURES_TSFRESH.drop(
    columns=["k"],
    errors="ignore"
)

# Merge k into FEATURES_TSFRESH
FEATURES_TSFRESH = FEATURES_TSFRESH.merge(
    k_lookup,
    on="Sample",
    how="left",
    validate="many_to_one"
)

# Check if k was successfully added
print(FEATURES_TSFRESH[["Sample", "k"]].drop_duplicates())

In [ ]:
legend_col = "Sample"
y_col = "k"

plots_per_fig = 9
n_cols = 3
n_rows = 3


# --------------------------------------------------
# Prepare data
# --------------------------------------------------

plot_df = FEATURES_TSFRESH.copy()

# Replace infinite values with NaN
plot_df = plot_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Ensure k is numeric
plot_df[y_col] = pd.to_numeric(
    plot_df[y_col],
    errors="coerce"
)

# Log scale requires positive k values
plot_df = plot_df[
    plot_df[y_col] > 0
].copy()


# --------------------------------------------------
# Select numeric tsfresh features
# --------------------------------------------------

excluded_cols = [
    "Sample",
    "Trial",
    "index",
    "k",
    "elbow_time"
]

candidate_features = [
    col
    for col in plot_df.select_dtypes(
        include=np.number
    ).columns
    if col not in excluded_cols
]


# Drop feature columns containing any NaN
features = [
    col
    for col in candidate_features
    if plot_df[col].notna().all()
]

dropped_features = [
    col
    for col in candidate_features
    if col not in features
]

print(
    f"Features retained: {len(features)}"
)

print(
    f"Features removed because of NaN: "
    f"{len(dropped_features)}"
)


# --------------------------------------------------
# Create one consistent color per material
# --------------------------------------------------

groups = (
    plot_df[legend_col]
    .dropna()
    .unique()
)

cmap = plt.get_cmap(
    "tab20",
    len(groups)
)

material_colors = {
    group: cmap(i)
    for i, group in enumerate(groups)
}


# --------------------------------------------------
# Plot k against each tsfresh feature
# --------------------------------------------------

n_figures = math.ceil(
    len(features) / plots_per_fig
)

for fig_num in range(n_figures):

    start = fig_num * plots_per_fig
    end = start + plots_per_fig

    current_features = features[start:end]

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 15)
    )

    axes = axes.flatten()

    for ax, feature in zip(
        axes,
        current_features
    ):

        for group in groups:

            subset = plot_df[
                plot_df[legend_col] == group
            ].dropna(
                subset=[feature, y_col]
            )

            if subset.empty:
                continue

            ax.scatter(
                subset[feature],
                subset[y_col],
                color=material_colors[group],
                label=str(group),
                alpha=0.8,
                s=60
            )

        ax.set_xlabel(feature)

        ax.set_ylabel(
            "Thermal conductivity, k [W/(m·K)]"
        )

        ax.set_title(
            f"{feature} vs k"
        )

        # Logarithmic numeric y-axis
        ax.set_yscale("log")

        # Show ordinary numeric labels
        ax.yaxis.set_major_formatter(
            ScalarFormatter()
        )

        ax.grid(
            which="both",
            linestyle=":",
            alpha=0.5
        )


    # Hide unused axes in the final figure
    for ax in axes[len(current_features):]:
        ax.axis("off")


    # Shared legend with multiple materials per row
    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor=material_colors[group],
            markeredgecolor=material_colors[group],
            markersize=7,
            label=str(group)
        )
        for group in groups
    ]

    fig.legend(
        handles=legend_handles,
        title="Material",
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=min(8, len(groups)),
        fontsize=8
    )

    fig.suptitle(
        f"Thermal conductivity vs tsfresh features "
        f"— Figure {fig_num + 1}",
        fontsize=16
    )

    plt.tight_layout(
        rect=[0, 0.10, 1, 0.97]
    )

    plt.show()

## ESN/RC Features

In [ ]:
from reservoirpy.nodes import Reservoir

# ------------------------------------------------------------
# Prepare ESN input
# ------------------------------------------------------------

esn_data = DATA.copy()

if "diff" not in esn_data.columns:
    esn_data["diff"] = esn_data["Secondary"] - esn_data["Primary"]

esn_data["id"] = (
    esn_data["Sample"].astype(str)
    + "_trial_"
    + esn_data["Trial"].astype(int).astype(str)
)

input_cols = [
    "Primary",
    "Secondary",
    "diff",
]

esn_data = esn_data.replace([np.inf, -np.inf], np.nan)
esn_data = esn_data.dropna(subset=["id", "Time"] + input_cols)

esn_data = esn_data.sort_values(["id", "Time"]).reset_index(drop=True)


# ------------------------------------------------------------
# Global input scaling
# ------------------------------------------------------------

scaler = StandardScaler()

esn_data[input_cols] = scaler.fit_transform(
    esn_data[input_cols]
)


# ------------------------------------------------------------
# ESN feature extraction
# ------------------------------------------------------------

def extract_FEATURES_ESN_from_sequence(
    sequence,
    units=100,
    sr=0.9,
    lr=0.3,
    input_scaling=0.5,
    random_state=42,
):
    reservoir = Reservoir(
        units=units,
        sr=sr,
        lr=lr,
        input_scaling=input_scaling,
        seed=random_state,
    )

    states = reservoir.run(sequence)

    features = {}

    for state_idx in range(states.shape[1]):
        state_values = states[:, state_idx]

        features[f"esn_state_{state_idx}_mean"] = np.mean(state_values)
        features[f"esn_state_{state_idx}_std"] = np.std(state_values)
        features[f"esn_state_{state_idx}_min"] = np.min(state_values)
        features[f"esn_state_{state_idx}_max"] = np.max(state_values)
        features[f"esn_state_{state_idx}_final"] = state_values[-1]

    return features

In [ ]:
esn_feature_rows = []

for trial_id, subset in esn_data.groupby("id", sort=False):
    subset = subset.sort_values("Time").copy()

    sequence = subset[input_cols].to_numpy(dtype=float)

    if len(sequence) < 3:
        print(f"Skipped {trial_id}: too few observations")
        continue

    features = extract_FEATURES_ESN_from_sequence(
        sequence,
        units=20,
        sr=0.3,
        lr=0.1,
        input_scaling=0.05,
        random_state=42,
    )

    features["id"] = trial_id

    esn_feature_rows.append(features)


FEATURES_ESN = pd.DataFrame(esn_feature_rows)


# ------------------------------------------------------------
# Add labels / metadata
# ------------------------------------------------------------

labels = (
    esn_data[["id", "Sample", "Trial", "Label", "k" , "eff"]]
    .drop_duplicates()
)

FEATURES_ESN = FEATURES_ESN.merge(
    labels,
    on="id",
    how="left",
    validate="one_to_one",
)

FEATURES_ESN.head()

# Machine Learning

## 6-Fold Classification

This subsection can be skipped as 武井先生 is less interested on this process.

In [ ]:
def simple_multiclass_logistic_cv(
    df,
    target_col="Label",
    stratify_col="Label",
    random_state=42,
    n_splits=6,
    class_order=[
        "foam",
        "composite",
        "polymer",
        "ceramic",
        "metal",
    ],
):
    work_df = (
        df
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[target_col, stratify_col])
        .reset_index(drop=False)
        .rename(columns={"index": "original_index"})
    )

    excluded_cols = [
        "Sample",
        "Trial",
        "Label",
        "k",
        "index",
        "original_index",
        "elbow_time",
        "elbow_idx",
        "elbow_value",
        "is_elbow",
        "Elbow time",
        "Elbow value",
        "Elbow index",
        "eff",
    ]

    features = [
        col
        for col in work_df.select_dtypes(include=np.number).columns
        if col not in excluded_cols
    ]

    features = [
        col
        for col in features
        if work_df[col].notna().any()
    ]

    X = work_df[features]
    y = work_df[target_col]
    stratify_y = work_df[stratify_col]

    ordered_classes = [
        label
        for label in class_order
        if label in y.unique()
    ]

    print("Number of features:", len(features))
    print("Target column:", target_col)
    print("Stratify column:", stratify_col)
    print("Classes:", ordered_classes)

    print("\nClass counts:")
    print(y.value_counts())

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    cv_rows = []
    oof_rows = []

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, stratify_y),
        start=1,
    ):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    solver="lbfgs",
                    max_iter=5000,
                    C=1.0,
                    class_weight=None,
                    random_state=random_state,
                ),
            ),
        ])

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        report = classification_report(
            y_test,
            y_pred,
            labels=ordered_classes,
            zero_division=0,
            output_dict=True,
        )

        cv_rows.append({
            "fold": fold,
            "n_test": len(test_idx),
            "accuracy": report["accuracy"],
            "macro_precision": report["macro avg"]["precision"],
            "macro_recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_precision": report["weighted avg"]["precision"],
            "weighted_recall": report["weighted avg"]["recall"],
            "weighted_f1": report["weighted avg"]["f1-score"],
        })

        fold_oof = work_df.iloc[test_idx][
            [
                "original_index",
                "Sample",
                "Trial",
                target_col,
            ]
        ].copy()

        fold_oof["fold"] = fold
        fold_oof["y_true"] = y_test.values
        fold_oof["y_pred"] = y_pred

        oof_rows.append(fold_oof)

        print(f"\nFold {fold}")
        print("Test rows:", len(test_idx))
        print("Accuracy:", report["accuracy"])

    cv_results = pd.DataFrame(cv_rows)
    oof_predictions = pd.concat(oof_rows, axis=0).sort_values("original_index")

    # ------------------------------------------------------------
    # Verify each original row appears exactly once out-of-fold
    # ------------------------------------------------------------

    duplicate_count = oof_predictions["original_index"].duplicated().sum()
    missing_count = len(work_df) - oof_predictions["original_index"].nunique()

    print("\nOOF prediction check:")
    print("Rows in modeling data:", len(work_df))
    print("Rows with OOF predictions:", len(oof_predictions))
    print("Unique original rows predicted:", oof_predictions["original_index"].nunique())
    print("Duplicate OOF predictions:", duplicate_count)
    print("Missing OOF predictions:", missing_count)

    if duplicate_count != 0 or missing_count != 0:
        raise ValueError(
            "OOF prediction issue: some rows were predicted more than once or not predicted."
        )

    # ------------------------------------------------------------
    # Final performance from all out-of-fold predictions
    # ------------------------------------------------------------

    y_true_oof = oof_predictions["y_true"]
    y_pred_oof = oof_predictions["y_pred"]

    overall_report_dict = classification_report(
        y_true_oof,
        y_pred_oof,
        labels=ordered_classes,
        zero_division=0,
        output_dict=True,
    )

    overall_results = {
        "accuracy": overall_report_dict["accuracy"],
        "macro_precision": overall_report_dict["macro avg"]["precision"],
        "macro_recall": overall_report_dict["macro avg"]["recall"],
        "macro_f1": overall_report_dict["macro avg"]["f1-score"],
        "weighted_precision": overall_report_dict["weighted avg"]["precision"],
        "weighted_recall": overall_report_dict["weighted avg"]["recall"],
        "weighted_f1": overall_report_dict["weighted avg"]["f1-score"],
    }

    overall_results = pd.DataFrame([overall_results])

    print("\nPer-fold CV performance:")
    print(cv_results.round(4))

    print("\nFinal out-of-fold performance:")
    print(overall_results.round(4))

    print("\nOverall out-of-fold classification report:")
    print(
        classification_report(
            y_true_oof,
            y_pred_oof,
            labels=ordered_classes,
            zero_division=0,
        )
    )

    ConfusionMatrixDisplay.from_predictions(
        y_true_oof,
        y_pred_oof,
        labels=ordered_classes,
        display_labels=[
            str(label).replace("_", " ").title()
            for label in ordered_classes
        ],
        cmap="Blues",
        xticks_rotation=45,
    )

    plt.xlabel("Predicted material cluster")
    plt.ylabel("Actual material cluster")
    plt.title("Out-of-Fold Multiclass Logistic Regression")
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Train final model on all data after evaluation
    # ------------------------------------------------------------

    final_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                solver="lbfgs",
                max_iter=5000,
                C=1.0,
                class_weight=None,
                random_state=random_state,
            ),
        ),
    ])

    final_model.fit(X, y)

    return final_model, features, cv_results, overall_results, oof_predictions

### Feature: thermal parameters

In [ ]:
# ------------------------------------------------------------
# Select corrected feature set for multiclass logistic CV
# ------------------------------------------------------------

logistic_feature_cols = [
    # --------------------------------------------------------
    # Primary sensor
    # --------------------------------------------------------
    "primary_max_change",
    "primary_response_auc",
    "primary_early_window_rate",
    "primary_mid_window_rate",
    "primary_late_window_rate",
    "primary_max_abs_rate",
    "primary_rate_auc",
    "primary_rate_std",

    # --------------------------------------------------------
    # Secondary sensor
    # --------------------------------------------------------
    "secondary_max_change",
    "secondary_response_auc",
    "secondary_early_window_rate",
    "secondary_mid_window_rate",
    "secondary_late_window_rate",
    "secondary_max_abs_rate",
    "secondary_rate_auc",
    "secondary_rate_std",

    # --------------------------------------------------------
    # Primary - Secondary difference
    # --------------------------------------------------------
    "difference_max_change",
    "difference_response_auc",
    "difference_early_window_rate",
    "difference_mid_window_rate",
    "difference_late_window_rate",
    "difference_max_abs_rate",
    "difference_rate_auc",
    "difference_rate_std",

    # --------------------------------------------------------
    # Secondary-to-primary ratios
    # --------------------------------------------------------
    "secondary_early_rate_ratio",
    "secondary_mid_rate_ratio",
    "secondary_late_rate_ratio",
    "secondary_rate_auc_ratio",

    # --------------------------------------------------------
    # Cross-sensor derivative features
    # --------------------------------------------------------
    "mean_top_bottom_rate_difference",
    "max_top_bottom_rate_difference",
    "max_abs_top_bottom_rate_difference",
    "top_bottom_rate_difference_auc",
    "top_bottom_rate_difference_std",

    # --------------------------------------------------------
    # PDMS heat-flux features
    # --------------------------------------------------------
    "q_pdms_mean",
    "q_pdms_max",
    "q_pdms_min",
    "q_pdms_max_abs",
    "q_pdms_final",
    "q_pdms_auc",
    "q_pdms_early_window_rate",
    "q_pdms_mid_window_rate",
    "q_pdms_late_window_rate",
    "q_pdms_rate_auc",
    "q_pdms_rate_std",

    "primary_energy_density_max_change",
    "primary_energy_density_response_auc",

    "primary_power_density_early_window_rate",
    "primary_power_density_mid_window_rate",
    "primary_power_density_late_window_rate",
    "primary_power_density_max_abs_rate",
    "primary_power_density_rate_auc",

    "primary_q_app_early_window_rate",
    "primary_q_app_mid_window_rate",
    "primary_q_app_late_window_rate",
    "primary_q_app_max_abs_rate",
    "primary_q_app_rate_auc",
]


# ------------------------------------------------------------
# Keep only columns that exist in features_df
# ------------------------------------------------------------

logistic_feature_cols = [
    col
    for col in logistic_feature_cols
    if col in FEATURES_THERMAL.columns
]

model_df = FEATURES_THERMAL[
    logistic_feature_cols + ["Label", "Sample", "Trial"]
].copy()


# ------------------------------------------------------------
# Run multiclass logistic CV
# ------------------------------------------------------------

logistic_model, logistic_features, logistic_cv, logistic_overall, logistic_pred  = simple_multiclass_logistic_cv(
    df=model_df,
    target_col="Label",
    n_splits=6,
    random_state=42,
)

print(f"Number of logistic features used: {len(logistic_feature_cols)}")
print(logistic_feature_cols)

In [ ]:
logistic_overall

### Features: time-series

In [ ]:
logistic_model, logistic_features, logistic_cv, logistic_overall, logistic_pred  = simple_multiclass_logistic_cv(
    df=FEATURES_TSFRESH[['Primary__mean', 'Primary__median', 'Primary__minimum',
       'Primary__maximum', 'Primary__absolute_maximum',
       'Primary__standard_deviation', 'Primary__variance',
       'Primary__root_mean_square', 'Primary__mean_abs_change',
       'Primary__mean_change', 'Primary__skewness', 'Primary__kurtosis',
       'Primary__abs_energy', 'Primary__sum_values',
       'Primary__count_above_mean', 'Primary__count_below_mean',
       'Primary__longest_strike_above_mean',
       'Primary__longest_strike_below_mean',
       'Primary__linear_trend__attr_"slope"',
       'Primary__linear_trend__attr_"intercept"',
       'Primary__linear_trend__attr_"rvalue"',
       'Primary__linear_trend__attr_"pvalue"',
       'Primary__linear_trend__attr_"stderr"',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8',
       'Primary__cid_ce__normalize_True', 'Primary__cid_ce__normalize_False', 'Label', 'Sample', 'Trial']],
    target_col="Label",
    n_splits=6,
    random_state=42,
)

In [ ]:
logistic_overall

### Features: ESN

In [ ]:
logistic_model, logistic_features, logistic_cv, logistic_overall, logistic_pred  = simple_multiclass_logistic_cv(
    df=FEATURES_ESN[[c for c in FEATURES_ESN.columns if 'esn' in c] + ['Label', 'Sample', 'Trial']],
    target_col="Label",
    n_splits=6,
    random_state=42,
)

In [ ]:
logistic_overall

## 6-Fold Regression

In [ ]:
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred) & ~np.isclose(y_true, 0)

    if valid.sum() == 0:
        return np.nan

    return np.mean(np.abs((y_true[valid] - y_pred[valid]) / y_true[valid])) * 100


def safe_medape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred) & ~np.isclose(y_true, 0)

    if valid.sum() == 0:
        return np.nan

    return np.median(np.abs((y_true[valid] - y_pred[valid]) / y_true[valid])) * 100


def safe_smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.abs(y_true) + np.abs(y_pred)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & ~np.isclose(denominator, 0)
    )

    if valid.sum() == 0:
        return np.nan

    return np.mean(
        2 * np.abs(y_pred[valid] - y_true[valid]) / denominator[valid]
    ) * 100


def safe_nrmse(y_true, y_pred, method="range"):
    """
    Normalized RMSE.

    method="range":
        NRMSE = RMSE / (max(y_true) - min(y_true))

    method="mean":
        NRMSE = RMSE / mean(y_true)

    method="std":
        NRMSE = RMSE / std(y_true)
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
    )

    if valid.sum() == 0:
        return np.nan

    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]

    rmse = np.sqrt(
        mean_squared_error(
            y_true_valid,
            y_pred_valid,
        )
    )

    if method == "range":
        denominator = np.max(y_true_valid) - np.min(y_true_valid)

    elif method == "mean":
        denominator = np.mean(y_true_valid)

    elif method == "std":
        denominator = np.std(y_true_valid, ddof=1)

    else:
        raise ValueError(
            "method must be one of: 'range', 'mean', or 'std'"
        )

    if not np.isfinite(denominator) or np.isclose(denominator, 0):
        return np.nan

    return rmse / denominator

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse,
        "nrmse_range": safe_nrmse(y_true, y_pred, method="range"),
        "nrmse_mean": safe_nrmse(y_true, y_pred, method="mean"),
        "nrmse_std": safe_nrmse(y_true, y_pred, method="std"),
        "r2": r2_score(y_true, y_pred) if len(y_true) >= 2 else np.nan,
        "mape": safe_mape(y_true, y_pred),
        "medape": safe_medape(y_true, y_pred),
        "smape": safe_smape(y_true, y_pred),
        "max_error": max_error(y_true, y_pred),
    }

def simple_log_linear_regression_cv(
    df,
    target_col="k",
    eval_group_col="Label",
    stratify_col="Label",
    random_state=42,
    n_splits=6
):
    work_df = (
        df
        .copy()
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[target_col])
        .reset_index(drop=False)
        .rename(columns={"index": "original_index"})
    )

    # log1p target requires y >= 0
    work_df = work_df[work_df[target_col] >= 0].copy()

    excluded_cols = [
        "Sample",
        "Trial",
        "Label",
        "k",
        "eff",
        "index",
        "original_index",
        "elbow_time",
        "elbow_idx",
        "elbow_value",
        "is_elbow",
        "Elbow time",
        "Elbow value",
        "Elbow index",
    ]

    excluded_cols = list(set(excluded_cols + [target_col]))

    features = [
        col
        for col in work_df.select_dtypes(include=np.number).columns
        if col not in excluded_cols
    ]

    features = [
        col
        for col in features
        if work_df[col].notna().any()
    ]

    X = work_df[features]
    y = work_df[target_col]

    print("Number of features:", len(features))
    print("Target:", target_col)
    print("Evaluation group:", eval_group_col)
    print("Stratify column:", stratify_col)
    print(f"Model target transform: log1p({target_col})")
    print(f"Prediction inverse transform: expm1(predicted log1p({target_col}))")

    # ------------------------------------------------------------
    # Choose CV splitter
    # ------------------------------------------------------------

    if stratify_col is not None and stratify_col in work_df.columns:
        stratify_y = work_df[stratify_col]

        min_class_count = stratify_y.value_counts().min()

        if min_class_count < n_splits:
            raise ValueError(
                f"Cannot use n_splits={n_splits}. "
                f"The smallest class in {stratify_col} has only {min_class_count} samples."
            )

        cv = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )

        split_iterator = cv.split(X, stratify_y)

    else:
        cv = KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )

        split_iterator = cv.split(X, y)

    cv_rows = []
    group_rows = []
    oof_rows = []

    # ------------------------------------------------------------
    # 6-fold CV
    # ------------------------------------------------------------

    for fold, (train_idx, test_idx) in enumerate(split_iterator, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        base_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("regressor", Ridge(alpha=0.1)),
        ])

        model = TransformedTargetRegressor(
            regressor=base_model,
            func=np.log1p,
            inverse_func=np.expm1,
            check_inverse=False,
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_pred = np.clip(y_pred, 0, None)

        fold_metrics = regression_metrics(y_test, y_pred)

        cv_rows.append({
            "fold": fold,
            "n_test": len(test_idx),
            **fold_metrics,
        })

        fold_oof = work_df.iloc[test_idx][
            [
                "original_index",
                eval_group_col,
            ]
        ].copy()

        if "Sample" in work_df.columns:
            fold_oof["Sample"] = work_df.iloc[test_idx]["Sample"].values

        if "Trial" in work_df.columns:
            fold_oof["Trial"] = work_df.iloc[test_idx]["Trial"].values

        fold_oof["fold"] = fold
        fold_oof["y_true"] = y_test.values
        fold_oof["y_pred"] = y_pred

        oof_rows.append(fold_oof)

        print(f"MAE: {fold_metrics['mae']:.4f}")
        print(f"RMSE: {fold_metrics['rmse']:.4f}")
        print(f"NRMSE range: {fold_metrics['nrmse_range']:.4f}")
        print(f"NRMSE mean: {fold_metrics['nrmse_mean']:.4f}")
        print(f"NRMSE std: {fold_metrics['nrmse_std']:.4f}")
        print(f"R2: {fold_metrics['r2']:.4f}")
        print(f"MAPE: {fold_metrics['mape']:.4f}%")
        print(f"MedAPE: {fold_metrics['medape']:.4f}%")
        print(f"SMAPE: {fold_metrics['smape']:.4f}%")
        print(f"Max error: {fold_metrics['max_error']:.4f}")

        # Per-group fold metrics
        fold_eval = pd.DataFrame({
            "group": work_df.iloc[test_idx][eval_group_col].values,
            "y_true": y_test.values,
            "y_pred": y_pred,
        })

        for group_name, group_data in fold_eval.groupby("group"):
            group_metrics = regression_metrics(
                group_data["y_true"],
                group_data["y_pred"],
            )

            group_rows.append({
                "fold": fold,
                "group": group_name,
                "n": len(group_data),
                **group_metrics,
            })

    cv_results = pd.DataFrame(cv_rows)
    #group_cv_results = pd.DataFrame(group_rows)

    oof_predictions = (
        pd.concat(oof_rows, axis=0)
        .sort_values("original_index")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------
    # Verify each row has exactly one OOF prediction
    # ------------------------------------------------------------

    duplicate_count = oof_predictions["original_index"].duplicated().sum()
    missing_count = len(work_df) - oof_predictions["original_index"].nunique()

    print("\nOOF prediction check:")
    print("Rows in modeling data:", len(work_df))
    print("Rows with OOF predictions:", len(oof_predictions))
    print("Unique original rows predicted:", oof_predictions["original_index"].nunique())
    print("Duplicate OOF predictions:", duplicate_count)
    print("Missing OOF predictions:", missing_count)

    if duplicate_count != 0 or missing_count != 0:
        raise ValueError(
            "OOF prediction issue: some rows were predicted more than once or not predicted."
        )

    # ------------------------------------------------------------
    # Final performance from all OOF predictions
    # ------------------------------------------------------------

    overall_metrics = regression_metrics(
        oof_predictions["y_true"],
        oof_predictions["y_pred"],
    )

    overall_results = pd.DataFrame([overall_metrics])

    print("\nPer-fold CV performance:")
    print(cv_results.round(4))

    print("\nFinal out-of-fold performance:")
    print(overall_results.round(4))

    print("\nAverage performance per group from OOF predictions:")
    group_oof_results = []

    for group_name, group_data in oof_predictions.groupby(eval_group_col):
        group_metrics = regression_metrics(
            group_data["y_true"],
            group_data["y_pred"],
        )

        group_oof_results.append({
            "group": group_name,
            "n": len(group_data),
            **group_metrics,
        })

    group_oof_results = pd.DataFrame(group_oof_results)
    print(group_oof_results.round(4))

    # ------------------------------------------------------------
    # Actual vs predicted plot using OOF predictions
    # ------------------------------------------------------------
    plt.figure(figsize=(7, 6))

    label_col = eval_group_col  # usually "Label"

    groups = (
        oof_predictions[label_col]
        .dropna()
        .unique()
    )

    groups = sorted(groups)

    cmap = plt.get_cmap("tab10", len(groups))

    group_colors = {
        group: cmap(i)
        for i, group in enumerate(groups)
    }

    for group in groups:
        subset = oof_predictions[
            oof_predictions[label_col] == group
        ]

        plt.scatter(
            subset["y_true"],
            subset["y_pred"],
            label=str(group).replace("_", " ").title(),
            color=group_colors[group],
            alpha=0.8,
            s=60,
            edgecolor="none",
        )

    min_value = 0
    max_value = max(
        oof_predictions["y_true"].max(),
        oof_predictions["y_pred"].max(),
    )

    plt.plot(
        [min_value, max_value],
        [min_value, max_value],
        linestyle=":",
        color="black",
        linewidth=1,
        label="Ideal prediction",
    )

    plt.xscale("log")
    plt.yscale("log")

    plt.xlabel(f"Actual {target_col}")
    plt.ylabel(f"Predicted {target_col}")
    plt.title(f"6-Fold OOF Log-Linear Regression: Actual vs Predicted {target_col}")

    plt.legend(
        title="Material cluster",
        fontsize=8,
        title_fontsize=9,
        loc="best",
    )

    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Final model trained on all data after evaluation
    # ------------------------------------------------------------

    final_base_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=0.1)),
    ])

    final_model = TransformedTargetRegressor(
        regressor=final_base_model,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )

    final_model.fit(X, y)

    return (
        final_model,
        features,
        cv_results,
        overall_results,
        group_oof_results,
        oof_predictions,
    )

### Features: thermal parameters

#### Target: k

In [ ]:
# ------------------------------------------------------------
# Select corrected feature set for thermal conductivity regression
# ------------------------------------------------------------

regression_feature_cols = [
    # Primary sensor
    "primary_max_change",
    "primary_response_auc",
    "primary_early_window_rate",
    "primary_mid_window_rate",
    "primary_late_window_rate",
    "primary_max_abs_rate",
    "primary_rate_auc",
    "primary_rate_std",

    # Secondary sensor
    "secondary_max_change",
    "secondary_response_auc",
    "secondary_early_window_rate",
    "secondary_mid_window_rate",
    "secondary_late_window_rate",
    "secondary_max_abs_rate",
    "secondary_rate_auc",
    "secondary_rate_std",

    # Primary - Secondary difference
    "difference_max_change",
    "difference_response_auc",
    "difference_early_window_rate",
    "difference_mid_window_rate",
    "difference_late_window_rate",
    "difference_max_abs_rate",
    "difference_rate_auc",
    "difference_rate_std",

    # Secondary-to-primary ratios
    "secondary_early_rate_ratio",
    "secondary_mid_rate_ratio",
    "secondary_late_rate_ratio",
    "secondary_rate_auc_ratio",

    # Cross-sensor derivative features
    "mean_top_bottom_rate_difference",
    "max_top_bottom_rate_difference",
    "max_abs_top_bottom_rate_difference",
    "top_bottom_rate_difference_auc",
    "top_bottom_rate_difference_std",

    # PDMS heat-flux features
    "q_pdms_mean",
    "q_pdms_max",
    "q_pdms_min",
    "q_pdms_max_abs",
    "q_pdms_final",
    "q_pdms_auc",
    "q_pdms_early_window_rate",
    "q_pdms_mid_window_rate",
    "q_pdms_late_window_rate",
    "q_pdms_rate_auc",
    "q_pdms_rate_std",

    "primary_energy_density_max_change",
    "primary_energy_density_response_auc",

    "primary_power_density_early_window_rate",
    "primary_power_density_mid_window_rate",
    "primary_power_density_late_window_rate",
    "primary_power_density_max_abs_rate",
    "primary_power_density_rate_auc",

    "primary_q_app_early_window_rate",
    "primary_q_app_mid_window_rate",
    "primary_q_app_late_window_rate",
    "primary_q_app_max_abs_rate",
    "primary_q_app_rate_auc",
]


# ------------------------------------------------------------
# Keep only columns that exist in FEATURES_THERMAL
# ------------------------------------------------------------

regression_feature_cols = [
    col
    for col in regression_feature_cols
    if col in FEATURES_THERMAL.columns
]


# ------------------------------------------------------------
# Build regression dataframe
# ------------------------------------------------------------

regression_df = FEATURES_THERMAL[
    regression_feature_cols + ["k", "Label" , "eff"]
].copy()


# ------------------------------------------------------------
# Run log-linear regression CV
# ------------------------------------------------------------


regression_model, regression_features, regression_cv, regression_overall, regression_oof, regression_pred = (
    simple_log_linear_regression_cv(
        df=regression_df,
        target_col="k",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

print(f"Number of regression features used: {len(regression_features)}")
print(regression_features)

In [ ]:
regression_overall

In [ ]:
regression_oof

#### Target: eff

In [ ]:
# ------------------------------------------------------------
# Run log-linear regression CV
# ------------------------------------------------------------

regression_model, regression_features, regression_cv, regression_overall, regression_oof, regression_pred = (
    simple_log_linear_regression_cv(
        df=regression_df,
        target_col="eff",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

print(f"Number of regression features used: {len(regression_features)}")
print(regression_features)

In [ ]:
regression_overall

In [ ]:
regression_oof

### Features: time-series

#### Target: k

In [ ]:
log_lin_model, log_lin_features, log_lin_cv, log_lin_overall, log_lin_off, log_lin_pred = (
    simple_log_linear_regression_cv(
        df=FEATURES_TSFRESH[[ 'Primary__mean', 'Primary__median', 'Primary__minimum',
       'Primary__maximum', 'Primary__absolute_maximum',
       'Primary__standard_deviation', 'Primary__variance',
       'Primary__root_mean_square', 'Primary__mean_abs_change',
       'Primary__mean_change', 'Primary__skewness', 'Primary__kurtosis',
       'Primary__abs_energy', 'Primary__sum_values',
       'Primary__count_above_mean', 'Primary__count_below_mean',
       'Primary__longest_strike_above_mean',
       'Primary__longest_strike_below_mean',
       'Primary__linear_trend__attr_"slope"',
       'Primary__linear_trend__attr_"intercept"',
       'Primary__linear_trend__attr_"rvalue"',
       'Primary__linear_trend__attr_"pvalue"',
       'Primary__linear_trend__attr_"stderr"',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8',
       'Primary__cid_ce__normalize_True', 'Primary__cid_ce__normalize_False', 'k', 'Label']],
        target_col="k",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

print(f"Number of regression features used: {len(log_lin_features)}")
print(log_lin_features)

In [ ]:
log_lin_overall

In [ ]:
log_lin_off

#### Target: eff

In [ ]:
log_lin_model, log_lin_features, log_lin_cv, log_lin_overall, log_lin_off, log_lin_pred = (
    simple_log_linear_regression_cv(
        df=FEATURES_TSFRESH[[ 'Primary__mean', 'Primary__median', 'Primary__minimum',
       'Primary__maximum', 'Primary__absolute_maximum',
       'Primary__standard_deviation', 'Primary__variance',
       'Primary__root_mean_square', 'Primary__mean_abs_change',
       'Primary__mean_change', 'Primary__skewness', 'Primary__kurtosis',
       'Primary__abs_energy', 'Primary__sum_values',
       'Primary__count_above_mean', 'Primary__count_below_mean',
       'Primary__longest_strike_above_mean',
       'Primary__longest_strike_below_mean',
       'Primary__linear_trend__attr_"slope"',
       'Primary__linear_trend__attr_"intercept"',
       'Primary__linear_trend__attr_"rvalue"',
       'Primary__linear_trend__attr_"pvalue"',
       'Primary__linear_trend__attr_"stderr"',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_0.2__ql_0.0',
       'Primary__change_quantiles__f_agg_"mean"__isabs_False__qh_1.0__ql_0.8',
       'Primary__change_quantiles__f_agg_"mean"__isabs_True__qh_1.0__ql_0.8',
       'Primary__cid_ce__normalize_True', 'Primary__cid_ce__normalize_False', 'eff', 'Label']],
        target_col="eff",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

print(f"Number of regression features used: {len(log_lin_features)}")
print(log_lin_features)

In [ ]:
log_lin_overall

In [ ]:
log_lin_off

### Features: ESN

Additional context:
- Initially, the regression target was set using litereature values of a material's known thermal conductivity $`\kappa`$. 
- However, the time series acquired from the sensors are more represented by thermal effusivity, $`e`$.
- Where $`e = \sqrt{\kappa\cdot\rho\cdot c_p}`$ and $`\rho`$ and $`c_p`$ are density and specific heat capacity, respectively. 
- Currently, the target effusivity of the material is set using literature values since $`\kappa`$ , $`\rho`$ , and $`c_p`$ are material constants (assumed to be constant for now for simplicity).
- It shows that the accuracy of the model increases when the target is thermal effusivity instead of thermal conductivity, hence subsections `Target: k` and `Target: eff` exist for comparative confirmation.

#### Target: k

In [ ]:
log_esn_model, log_esn_features, log_esn_cv, log_esn_overall, log_esn_off, log_esn_pred = (
    simple_log_linear_regression_cv(
        df=FEATURES_ESN[[c for c in FEATURES_ESN.columns if 'esn' in c] + ['k', 'Label']],
        target_col="k",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

In [ ]:
log_esn_overall

In [ ]:
log_esn_off

#### Target: eff

In [ ]:
log_esn_model, log_esn_features, log_esn_cv, log_esn_overall, log_esn_off, log_esn_pred = (
    simple_log_linear_regression_cv(
        df=FEATURES_ESN[[c for c in FEATURES_ESN.columns if 'esn' in c] + ['eff', 'Label']],
        target_col="eff",
        eval_group_col="Label",
        n_splits=6,
        random_state=42,
    )
)

In [ ]:
log_esn_overall

In [ ]:
log_esn_off